FPGrowth Algorithm

load the data


In [0]:
df_spark = spark.read.csv(
    '/Volumes/workspace/testschema/testunitycolume/Groceries_dataset.csv',
    header=True,
    inferSchema=True
)
display(df_spark.head(10))

Preprocess data

In [0]:
from pyspark.sql import functions as F

# Assume your DataFrame is called df and column is 'Date'

df_clean = df_spark.withColumn(
    "Date_modified",
    F.when(
        # If 3rd character is '-' (means format is DD-MM-YYYY)
        F.substring("Date", 3, 1) == "-",
        F.date_format(
            F.to_date(F.col("Date"), "dd-MM-yyyy"), "yyyy-MM-dd"
        )
    ).otherwise(
        # Else assume format is YYYY-MM-DD
        F.date_format(
            F.to_date(F.col("Date"), "yyyy-MM-dd"), "yyyy-MM-dd"
        )
    )
)

display(df_clean.head(10))


In [0]:
from pyspark.sql.functions import collect_set

# Group items by Member_number and Date to form transactions
transactions_df = df_clean.groupBy(
    'Member_number', 'Date_modified'
).agg(
    collect_set('itemDescription').alias('items')
)

display(transactions_df.head(10))

In [0]:
from pyspark.ml.fpm import FPGrowth

fpGrowth = FPGrowth(
    itemsCol='items',
    minSupport=0.02,
    minConfidence=0.3
)

model = fpGrowth.fit(transactions_df)

display(model.freqItemsets)
display(model.associationRules)